# Participación regional de la demanda industrial

Procesa `Insumos/Datos_Industrial_Regionalizado.xlsx` (una hoja por región), repara los nombres de tecnología truncados por el exporte de origen, limpia la ruta de tecnología (separando solo por `\`), consolida en formato largo y calcula dos participaciones porcentuales regionales:

1. Por **tecnología completa** (`Uso\fuel\eficiencia`) y `Año`.
2. Agrupada exclusivamente por **FUEL** (segundo segmento de la ruta limpia) y `Año`.

Se omite el año 2021 y los totales nacionales nulos o cero producen participación 0.

In [ ]:
import pandas as pd

pd.set_option('display.max_rows', 200)

In [ ]:
# --- Configuración ---
INPUT_PATH = "Insumos/Salida Compilada - Residencial (Energía útil) - Sin subtotales-2.xlsx"
OUTPUT_PATH = "Insumos/Participacion_Regional_Residencial.xlsx"

HEADER_ROW = 5          # fila 0-indexada donde está el encabezado real ("Branch", 2021, 2022, ...)
ANIO_EXCLUIDO = 2021
COL_RUTA = "Branch"

## 1. Lectura de todas las hojas (regiones)

In [ ]:
hojas = pd.read_excel(INPUT_PATH, sheet_name=None, skiprows=HEADER_ROW)
print("Regiones encontradas:", list(hojas.keys()))

primera = next(iter(hojas.values()))
print("Columnas de ejemplo:", primera.columns.tolist())

## 2. Limpieza de la ruta de tecnología y filtro de años

Toda la limpieza se hace separando por el carácter `\` (sin límites de longitud ni índices fijos):

1. **Reparación de truncamiento**: el exporte de origen limita la ruta a 100 caracteres, por lo que algunos segmentos finales llegan cortados (ej. `Mejor eficiencia_internaciona`). Se construye un mapa de reparación comparando cada segmento final contra los nombres canónicos completos (`Eficiencia_existente`, `Mejor eficiencia_Colombia`, `Mejor eficiencia_internacional`) y se restaura el texto completo.
2. Se elimina el primer segmento (`Subsector {Código}`), dejando la ruta como `Uso\fuel\eficiencia`.
3. Se descartan las filas `Total`, la columna `2021` y cualquier columna auxiliar como `Total`.
4. Se agrupan (sumando) las rutas que queden duplicadas dentro de cada hoja.

In [ ]:
def construir_mapa_reparacion(hojas: dict) -> dict:
    """Mapa {segmento_final_truncado: nombre_canonico_completo}.

    El origen limita la ruta a 100 caracteres y puede cortar el último
    segmento (ej. 'Mejor eficiencia_internaciona'). Un segmento es canónico
    si NO es prefijo estricto de otro segmento observado; cada segmento
    truncado se mapea al único canónico que lo contiene como prefijo.
    """
    finales = set()
    for df in hojas.values():
        finales |= {str(r).split("\\")[-1] for r in df[COL_RUTA].dropna()}

    canonicos = [s for s in finales if not any(o != s and o.startswith(s) for o in finales)]

    mapa = {}
    for seg in finales:
        candidatos = [c for c in canonicos if c.startswith(seg)]
        if len(candidatos) == 1:
            mapa[seg] = candidatos[0]
    return mapa


MAPA_REPARACION = construir_mapa_reparacion(hojas)
reparados = {k: v for k, v in MAPA_REPARACION.items() if k != v}
print("Segmentos truncados reparados:")
for trunc, completo in sorted(reparados.items()):
    print(f"  {trunc!r} -> {completo!r}")


def limpiar_ruta(ruta: str) -> str:
    """Repara el segmento final truncado y quita el primer segmento
    'Subsector {Codigo}'. Solo usa separación por backslash, sin cortes
    por longitud ni índices fijos de caracteres."""
    partes = str(ruta).split("\\")
    partes[-1] = MAPA_REPARACION.get(partes[-1], partes[-1])
    return "\\".join(partes[1:]) if len(partes) > 1 else partes[0]


def procesar_hoja(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Descartar filas sin ruta o filas de totales (ej. Branch == 'Total')
    df = df[df[COL_RUTA].notna()]
    df = df[df[COL_RUTA].astype(str).str.split("\\").str.len() >= 2]

    # Columnas de año: numéricas y dentro del rango válido (excluye 2021 y cualquier 'Total')
    year_cols = [
        c for c in df.columns
        if isinstance(c, (int, float)) and not pd.isna(c) and int(c) != ANIO_EXCLUIDO
    ]

    df["Tecnologia"] = df[COL_RUTA].apply(limpiar_ruta)

    df = df[["Tecnologia"] + year_cols]
    df.columns = ["Tecnologia"] + [int(c) for c in year_cols]

    # Agrupar rutas duplicadas tras la limpieza, sumando valores por año
    df = df.groupby("Tecnologia", as_index=False).sum(numeric_only=True)
    return df


hojas_limpias = {region: procesar_hoja(df) for region, df in hojas.items()}
hojas_limpias["Antioquia"].head()

## 3. Formato largo y consolidación en un DataFrame maestro

In [ ]:
def a_formato_largo(region: str, df: pd.DataFrame) -> pd.DataFrame:
    year_cols = [c for c in df.columns if c != "Tecnologia"]
    largo = df.melt(id_vars="Tecnologia", value_vars=year_cols, var_name="Anio", value_name="Valor")
    largo.insert(0, "Region", region)
    return largo


df_maestro = pd.concat(
    [a_formato_largo(region, df) for region, df in hojas_limpias.items()],
    ignore_index=True,
)
df_maestro["Anio"] = df_maestro["Anio"].astype(int)
df_maestro["Valor"] = pd.to_numeric(df_maestro["Valor"], errors="coerce").fillna(0)

print(df_maestro.shape)
df_maestro.head()

## 4. Total nacional por Tecnología y Año

In [ ]:
df_total_nacional = (
    df_maestro.groupby(["Tecnologia", "Anio"], as_index=False)["Valor"]
    .sum()
    .rename(columns={"Valor": "Total_Nacional"})
)
df_total_nacional.head()

## 5. Cálculo de participación porcentual por región

In [ ]:
df_participacion = pd.merge(df_maestro, df_total_nacional, on=["Tecnologia", "Anio"], how="left")

# Manejo de división por cero / nulos: si el total nacional es 0 (o NaN), la participación es 0
df_participacion["Total_Nacional"] = df_participacion["Total_Nacional"].fillna(0)
df_participacion["Participacion"] = 0.0
mask_valido = df_participacion["Total_Nacional"] != 0
df_participacion.loc[mask_valido, "Participacion"] = (
    df_participacion.loc[mask_valido, "Valor"] / df_participacion.loc[mask_valido, "Total_Nacional"]
)

df_participacion = df_participacion.sort_values(["Tecnologia", "Anio", "Region"]).reset_index(drop=True)
df_participacion.head(20)

## 6. Salida: regiones como columnas, indexado por Tecnología y Año

In [ ]:
df_salida = df_participacion.pivot_table(
    index=["Tecnologia", "Anio"],
    columns="Region",
    values="Participacion",
    fill_value=0,
)
df_salida.columns.name = None
df_salida = df_salida.reset_index()
df_salida.head(20)

In [ ]:
# Verificación: la suma de participaciones por Tecnologia-Anio debe ser ~1 (o 0 si el total nacional es 0)
region_cols = [c for c in df_salida.columns if c not in ("Tecnologia", "Anio")]
suma_check = df_salida[region_cols].sum(axis=1)
filas_invalidas = ((suma_check - 1).abs() > 1e-6) & (suma_check.abs() > 1e-6)
print(f"Filas con suma de participación distinta de 1 (y != 0): {filas_invalidas.sum()} de {len(df_salida)}")

## 7. Participación regional agrupada exclusivamente por FUEL

La ruta limpia tiene la forma `Uso\fuel\eficiencia`. Se separa de nuevo por `\` y se extrae **únicamente el segundo elemento** (el combustible, FUEL). Ej.: de `Aire acondicionado\Auto_cogeneración\Mejor eficiencia_internacional` se extrae `Auto_cogeneración`.

Se agrupan los valores por `Region`, `Fuel` y `Anio`, se calcula el total nacional a nivel de FUEL y la participación de cada región por combustible y año.

In [ ]:
# Extraer el FUEL: segundo elemento de la ruta limpia 'Uso\fuel\eficiencia'
df_fuel = df_maestro.copy()
df_fuel["Fuel"] = df_fuel["Tecnologia"].str.split("\\").str[1]

fuels_nulos = df_fuel["Fuel"].isna().sum()
if fuels_nulos:
    print(f"Advertencia: {fuels_nulos} filas sin segundo segmento (se descartan)")
    df_fuel = df_fuel[df_fuel["Fuel"].notna()]

print("Combustibles encontrados:", sorted(df_fuel["Fuel"].unique()))

# Agrupar exclusivamente por Region, Fuel y Anio
df_fuel = df_fuel.groupby(["Region", "Fuel", "Anio"], as_index=False)["Valor"].sum()
df_fuel.head()

In [ ]:
# Total nacional a nivel de FUEL y participación regional
df_total_fuel = (
    df_fuel.groupby(["Fuel", "Anio"], as_index=False)["Valor"]
    .sum()
    .rename(columns={"Valor": "Total_Nacional"})
)

df_participacion_fuel = pd.merge(df_fuel, df_total_fuel, on=["Fuel", "Anio"], how="left")

# Manejo de división por cero / nulos: si el total nacional es 0 (o NaN), la participación es 0
df_participacion_fuel["Total_Nacional"] = df_participacion_fuel["Total_Nacional"].fillna(0)
df_participacion_fuel["Participacion"] = 0.0
mask_valido = df_participacion_fuel["Total_Nacional"] != 0
df_participacion_fuel.loc[mask_valido, "Participacion"] = (
    df_participacion_fuel.loc[mask_valido, "Valor"]
    / df_participacion_fuel.loc[mask_valido, "Total_Nacional"]
)

df_participacion_fuel = (
    df_participacion_fuel.sort_values(["Fuel", "Anio", "Region"]).reset_index(drop=True)
)
df_participacion_fuel.head(20)

In [ ]:
# Salida por FUEL: regiones como columnas, indexado por Fuel y Año
df_salida_fuel = df_participacion_fuel.pivot_table(
    index=["Fuel", "Anio"],
    columns="Region",
    values="Participacion",
    fill_value=0,
)
df_salida_fuel.columns.name = None
df_salida_fuel = df_salida_fuel.reset_index()

# Verificación: la suma de participaciones por Fuel-Anio debe ser ~1 (o 0 si el total nacional es 0)
region_cols_fuel = [c for c in df_salida_fuel.columns if c not in ("Fuel", "Anio")]
suma_check_fuel = df_salida_fuel[region_cols_fuel].sum(axis=1)
filas_invalidas_fuel = ((suma_check_fuel - 1).abs() > 1e-6) & (suma_check_fuel.abs() > 1e-6)
print(f"Filas con suma de participación distinta de 1 (y != 0): {filas_invalidas_fuel.sum()} de {len(df_salida_fuel)}")
df_salida_fuel.head(20)

## 8. Exportar resultado

Un solo archivo Excel con cuatro hojas: participación por tecnología completa (`Uso\fuel\eficiencia`) y participación agrupada solo por FUEL, cada una en formato pivote (regiones como columnas) y plano.

In [ ]:
with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    # Archivo/cálculo 1: participación por tecnología completa (Uso\fuel\eficiencia)
    df_salida.to_excel(writer, sheet_name="Participacion_Regional", index=False)
    df_participacion.to_excel(writer, sheet_name="Participacion_Plano", index=False)
    # Archivo/cálculo 2: participación agrupada exclusivamente por FUEL
    df_salida_fuel.to_excel(writer, sheet_name="Participacion_Fuel", index=False)
    df_participacion_fuel.to_excel(writer, sheet_name="Participacion_Fuel_Plano", index=False)

print(f"Archivo exportado en: {OUTPUT_PATH}")

# Verificación final: ninguna cadena debe quedar truncada
sospechosas = [t for t in df_salida["Tecnologia"].unique()
               if t.split("\\")[-1] not in ("Eficiencia_existente", "Mejor eficiencia_Colombia", "Mejor eficiencia_internacional")]
print(f"Tecnologías con segmento final no canónico: {len(sospechosas)}")
if sospechosas:
    print(sospechosas[:10])